# Supplementary Figure 7 — rats

The submitted Extended Data Fig. 5 (24 sessions, 5 rats): a as submitted, b and c on non-overlapping windows, d on all windows (as submitted). d2 = d2 (2024 Euclidean, AR p = 20), labelled "d2". Rat sessions mix 40–80 ms bins and 30 / 60 s windows (b: the 30 s files only). Panels carry stars only (*p < 0.05, **p < 0.01, ***p < 0.001, ****p < 0.0001); session counts are in the figure legend. Every test is per session.

**Wake**: b = every wake window, θ / non-θ by the state at the window centre (as mouse Fig 1e); c and d = task wake as submitted and as the later rat figures — wake windows inside task epochs, which in the rats are cheeseboard and linear-track epochs (not a T-maze); c and d label θ / non-θ by the whole window (θ intervals eroded / dilated by 15 s, `generate_pairs_df.ipynb`).

- **a** the submitted example trace (HP18 day 36), cut from `Extended Data Fig5.svg` unchanged.
- **b** d2 by state with the boxes inset (the final Fig 1e design), per-session medians, session-paired signed-rank: wake non-θ vs θ; NREM vs REM. Sleep windows exclude the PFC-silencing epochs (`manipulation != "pfc_silence"`), as in cell 15 (c, d); a second arm applying the string `manipulation != "pcf_silence"` (which matches every row) is kept for comparison.
- **c** raw d2 vs MUA (z within session × Wake/Sleep) (as Supplementary Fig 2f), task wake, (session, state) cells with ≥ 4 windows. Legend = window-weighted r (random-effects meta-analysis of the per-session Fisher z), no stars; its p and the mixed model's p (random intercept + slope) go in the written legend. Raw d2 levels differ between sessions with the bin size; the per-session r does not.
- **d** spontaneous firing-rate range (renamed from "dynamic range") by d2 bin, all windows, task wake, the notebook's one-sided Stouffer test, reproduced exactly.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('.'))   # notebooks/figures_revision
import numpy as np
import pandas as pd
import final_figs as F                     # paths, save/copy helpers; adds the revision folder to sys.path

## Data — two runs of the submitted rat notebook; the per-session tables and tests

In [ ]:
import re
import tempfile
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import ed_nonoverlap as E
import fig1e_scatter_d2raw as SC                       # meta_r: per-session r pooled, weighted by window count

# Supplementary Fig 7 = the submitted Extended Data Fig. 5 (rats; 24 sessions, 5 rats):
#   a  example trace (HP18 day 36), the submitted panel cut from Extended Data Fig5.svg
#   b  d2 distributions + boxes by state, non-overlapping windows        Previous_Rv1 FigS5_S6_rats cell 7
#   c  d2 (z) vs MUA (z), non-overlapping windows                         cell 17
#   d  spontaneous firing-rate range by d2 bin, ALL windows (as submitted) cell 20
# Wake: b = EVERY wake window, theta / non-theta by the state at the window centre (as the mouse Fig 1e); c and d =
# TASK wake as submitted (cell 15: wake windows inside task epochs -- in the rats cheeseboard and linear-track
# epochs, not a T-maze), as the later rat figures. c: RAW d2 vs MUA (z) (as Supplementary Fig 2f); the rat sessions
# mix 40-80 ms bins and 30 / 60 s windows, so raw d2 levels differ between sessions -- the per-session r is
# unaffected.
NB = "FigS5_S6_rats"
LOG = os.path.join(F.REPO, "data", "revision", "log")
TB = os.path.join(LOG, "figs7b_rat_d2_windows_30s_disjoint.csv")
TC = os.path.join(LOG, "figs7c_rat_scatter_pairs_disjoint.csv")
TD = os.path.join(LOG, "figs7d_rat_frrange_perfile_rho_allwindows.csv")
MIN_C = 4                                              # c: (session, state) cells with >= 4 windows (the meta minimum)


def rv1_run(keys, exports=(), patches=(), keep_all=()):
    """One run of the rat notebook as ed_nonoverlap runs it: figures saved under `keys` captured, DataFrames written
    at top-level source lines (exports: (line, expression, csv)), extra asserted patches; the tables named in
    keep_all are read with every window (the submitted analysis), the rest thinned as usual. Returns (figures, log)."""
    pat = list(patches) + [(a, a + f"\n{v}.to_csv(r'{F.assert_local_path(o)}', index=False)") for a, v, o in exports]
    saved = E.PATCH.get(NB), E.OUTROOT, E.THIN
    E.PATCH[NB], E.OUTROOT = pat, Path(tempfile.mkdtemp(prefix="rv1_figs7_"))
    E.THIN = {k: v for k, v in E.THIN.items() if k not in keep_all}
    try:
        figs = F.capture_figures(lambda: E.run_notebook(NB), keys)
        return figs, str(E.OUTROOT / f"{NB}.log")
    finally:
        (E.PATCH[NB], E.OUTROOT, E.THIN) = saved


# ---- run A: non-overlapping windows -> b's table, c's table and c's figure
CAP, _ = rv1_run(["figS5_C_rat_NREM_REM_MUA"],
                 exports=[('df_30s_only = df_binned[df_binned["window"] == "30s"].copy()', "df_30s_only", TB),
                          ("task_wake = z_pairs[task_wake_mask].copy()", "z_pairs", TC)],
                 patches=[("keep_keys = counts[counts >= MIN_N].index",
                           f"keep_keys = counts[counts >= {MIN_C}].index  # S7: >= {MIN_C} disjoint windows per cell"),
                          ('out["d2_z"]    = out.groupby(list(by), dropna=False)["d2"].transform(safe_z)',
                           'out["d2_z"]    = out["d2"].astype(float)   # S7c: raw d2 drawn; MUA stays z')])
# ---- run B: every window (d as submitted) -> d's figures, its printed tests, and the per-session rho
if os.path.exists(TD):
    os.remove(TD)
ROW = '    per_file_rho = res["per_file"][["__file__", "rho"]]'
CAPD, LOGD = rv1_run(["figS5_rat_D__NREM_vs_REM_dyn_by_d2", "figS5_rat_D_THETA_vs_nonTHETA_dyn_by_d2"],
                     keep_all=("rat_d2_mua_pairs.csv",),
                     patches=[(ROW, ROW + f'\n    per_file_rho.assign(label=label).to_csv(r"{TD}", mode="a", '
                                         f'header=not os.path.exists(r"{TD}"), index=False)')])
CAP.update(CAPD)

# ---- b: per-session medians, session-paired signed-rank (as Fig 1e)
B = pd.read_csv(TB, low_memory=False)
B["session"] = B.animal.astype(str) + "_" + B.day.astype(str)
manip = B.manipulation.astype(str).str.lower()
# Sleep: NREM / REM windows outside PFC-silencing epochs (manipulation != "pfc_silence"), as in cell 15 (c, d).
SLEEP_FIX = B[B.state.isin(["NREM", "REM"]) & (manip != "pfc_silence")]
WAKE_B = B[B.state.isin(["THETA", "nonTHETA"])]          # every wake window; the submitted panel: env != sleep


def paired_sessions(d, a, b):
    m = d.groupby(["session", "state"]).d2.median().unstack().dropna(subset=[a, b])
    diff = m[a] - m[b]
    up = diff.median() > 0
    return dict(n=len(m), n_dir=int((diff > 0).sum()) if up else int((diff < 0).sum()), direction="↑" if up else "↓",
                median_diff=float(diff.median()), p=float(stats.wilcoxon(m[a], m[b]).pvalue))


ST = []
for arm, d, a, b in (("Wake", WAKE_B, "nonTHETA", "THETA"), ("Sleep", SLEEP_FIX, "NREM", "REM")):
    s = paired_sessions(d, a, b)
    ST.append(dict(panel="b", arm=arm, contrast=f"{a} - {b}", windows=len(d),
                   **{f"windows_{k}": int((d.state == k).sum()) for k in (a, b)}, **s))
    print(f"b {arm}: {a} vs {b}, two-sided paired Wilcoxon signed-rank p = {s['p']:.2g}; n = {s['n']} sessions")

# ---- c: per-state window-weighted r (random-effects meta of per-session Fisher z), as S2f
C = (pd.read_csv(TC, low_memory=False).drop(columns=["d2_z", "mua30"])          # raw d2 (d2_z is raw by the patch)
       .rename(columns={"mua30_z": "mua_z"}))
STATES_C = {"Wake non-θ": C[(C.group == "Wake non-θ") & C.epoch_name.notna()],
            "Wake θ": C[(C.group == "Wake θ") & C.epoch_name.notna()],
            "NREM": C[C.group == "NREM"], "REM": C[C.group.isin(["REM", "REM4"])]}
SCS = {}
for st, d in STATES_C.items():
    d = d.dropna(subset=["d2", "mua_z"]).copy()
    meta = SC.meta_r(d, min_n=MIN_C)
    SCS[st] = dict(panel="c", arm=st, **meta)
    print(f"c {st}: per-session Pearson r pooled by random-effects meta-analysis, r = {meta['r_re']:+.2f}, "
          f"p = {meta['p_re']:.2g}")

# ---- d: the notebook's own printed tests (per-file Spearman, Stouffer, one-sided "decreasing"), all windows
DYN = {}
for line in open(LOGD, encoding="utf-8"):
    m = re.match(r"(\S+): z_comb = ([-\d.]+), rho_wmean_fisher = ([-\d.]+), p_one_sided = (\S+), files = (\d+), "
                 r"animals = (\d+)", line.strip())
    if m:
        DYN[m.group(1)] = dict(z=float(m.group(2)), rho=float(m.group(3)), p=float(m.group(4).rstrip(",")),
                               files=int(m.group(5)), animals=int(m.group(6)))
for lab, s in DYN.items():
    print(f"d {lab}: per-session Spearman rho combined by Stouffer's method (one-sided), rho = {s['rho']:+.2f}, "
          f"p = {s['p']:.3g}")

## Panels a–d

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D

# ---- a: the submitted example trace (HP18 day 36), cut from Extended Data Fig. 5 unchanged; its letter "a" left out
ED5 = os.path.join(F.REPO, "reports", "new_planc", "nature", "Extended Data Fig5.svg")
F.crop_svg(ED5, "FigS7_A_rat_example_trace", 2.0, 6.4, 99.0, 46.2, drop_ids=["text334027-7-23"], unit_mm=1.0)
F.show(F.stem("FigS7_A_rat_example_trace") + ".png")

# ---- b: d2 distributions by state with the boxes inset -- the final Fig 1e panel's design (Fig1_E_d2_histogram_
# nonoverlap), on the rat table; every wake window, centre theta (as Fig 1e); sleep outside PFC-silencing epochs
ARMS = [("Wake", WAKE_B, ["THETA", "nonTHETA"], {"THETA": "#d95f0e", "nonTHETA": "#fdb863"}, ["θ", "non-θ"], ST[0]),
        ("Sleep", SLEEP_FIX, ["NREM", "REM"], {"NREM": "#7b3294", "REM": "#d7b5d8"}, ["NREM", "REM"], ST[1])]
F.final_style()
fig, axes = plt.subplots(1, 2, figsize=(76 * F.MM, 46.2 * F.MM), constrained_layout=True)
for ax, (title, d, order, pal, ticks, s) in zip(axes, ARMS):
    sns.kdeplot(data=d, x="d2", hue="state", hue_order=order[::-1], palette=pal, fill=True, alpha=0.2,
                common_norm=False, bw_adjust=0.5, linewidth=0.9, legend=False, ax=ax)
    ax.set_xlim(0, 0.25); ax.set_xticks([0, 0.1, 0.2]); ax.set_xticklabels(["0", "0.1", "0.2"])
    ax.set_xlabel("d2"); ax.set_ylabel("Probability density" if ax is axes[0] else "")
    ax.set_title(title, loc="left")
    ins = ax.inset_axes([0.6, 0.38, 0.38, 0.6])
    sns.boxplot(data=d, x="state", y="d2", order=order, hue="state", hue_order=order, palette=pal, legend=False,
                showfliers=False, width=0.6, linewidth=0.5, ax=ins)
    ins.set_xticks([0, 1]); ins.set_xticklabels(ticks, fontsize=5); ins.set_xlabel(""); ins.set_ylabel("d2" if ax is axes[0] else "", labelpad=1)   # right inset: would sit on the NREM curve
    q = [np.percentile(d.loc[d.state == k, "d2"], [25, 75]) for k in order]
    yb = 1.06 * max(d.loc[(d.state == k) & (d.d2 <= q3 + 1.5 * (q3 - q1)), "d2"].max() for k, (q1, q3) in zip(order, q))
    ins.set_ylim(0, yb * 1.2); ins.set_yticks([t for t in (0, 0.1, 0.2) if t <= yb * 1.2])
    ins.set_yticklabels([f"{t:g}" for t in ins.get_yticks()])
    ins.tick_params(labelsize=5, length=1.5, pad=1)
    ins.plot([0, 0, 1, 1], [yb * 0.97, yb, yb, yb * 0.97], color="k", lw=0.5)
    ins.text(0.5, yb * 1.02, f"{s['n_dir']}/{s['n']} {s['direction']} {F.stars(s['p'])}", ha="center", va="bottom",
             fontsize=5)                                                     # session-paired signed-rank
F.settle(fig)
F.drop_counts(fig)
F.show(F.save(fig, "FigS7_B_d2_by_state"))

# ---- c, d: the Supplementary Fig 2f geometry (one axes height for the scatter and the two range plots)
H, Bm, Tm = 58.0, 17.0, 7.0
AX_H = H - Bm - Tm


def place(fig, w_mm, lefts, width):
    fig.set_layout_engine("none")
    fig.set_size_inches(w_mm * F.MM, H * F.MM)
    for ax_, l in zip(fig.axes, lefts):
        ax_.set_position([l / w_mm, Bm / H, width / w_mm, AX_H / H])


def legend_below(ax_, handles, labels, gap_mm=6.6):
    ax_.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, -gap_mm / AX_H), frameon=False, fontsize=5,
               handlelength=1.6, handletextpad=0.4, borderaxespad=0, labelspacing=0.3)


# c: d2 (z) vs MUA (z), task wake, non-overlapping windows; the notebook's drawing and fit lines, its legend replaced
# by the window-weighted r (no stars: its p and the mixed model's p go in the written legend, as Supplementary Fig 2f)
fig = CAP["figS5_C_rat_NREM_REM_MUA"]
for lab_ in (getattr(fig, "_supylabel", None), getattr(fig, "_supxlabel", None)):   # the notebook's shared labels:
    if lab_ is not None:                                                         # each axes has its own
        lab_.set_visible(False)
F.restyle(fig, 108, H)
axw, axs = fig.axes[:2]
for ax_ in (axw, axs):
    ax_.set_xlabel("MUA (z, within session)")
    ax_.title.set_fontsize(6)
    if ax_.get_legend():
        ax_.get_legend().remove()
axw.set_ylabel("d2"); axs.set_ylabel("")
axw.set_title("Task wake\nWake θ + Wake non-θ", fontsize=6)    # rats: cheeseboard / linear track, not a T-maze


def entry(st, pal, lpal):
    s_ = SCS[st]                                                             # (the statistic: see the data cell)
    h = Line2D([0], [0], marker="o", ms=2.5, ls="--", lw=0.8, color=lpal[st], markerfacecolor=pal[st], markeredgecolor=pal[st])
    return h, f"{st}: r = {s_['r_re']:+.2f}"


GROUPS = ((axw, ("Wake non-θ", "Wake θ"), SC.WAKE_PAL, SC.WAKE_LINE), (axs, ("NREM", "REM"), SC.SLEEP_PAL, SC.SLEEP_LINE))
place(fig, 108, (9.0, 61.0), 45.0)
for ax_, sts, pal, lpal in GROUPS:
    hl = [entry(st, pal, lpal) for st in sts]
    legend_below(ax_, [h for h, _ in hl], [l for _, l in hl])
F.show(F.save(fig, "FigS7_C_d2_vs_mua", tight=False))

# d: spontaneous firing-rate range by d2 bin, every window (as submitted), task wake; the notebook's figures, renamed
# from "dynamic range", legend with the notebook's own one-sided Stouffer test (stars)
NAME = {"THETA": "Wake θ", "non-THETA": "Wake non-θ"}
for key, fid, title in (("figS5_rat_D__NREM_vs_REM_dyn_by_d2", "FigS7_D_firing_rate_range_nrem_rem", "Firing-rate range\n(NREM / REM)"),
                        ("figS5_rat_D_THETA_vs_nonTHETA_dyn_by_d2", "FigS7_D_firing_rate_range_theta", "Firing-rate range\n(task wake θ / non-θ)")):
    fig = CAP[key]
    F.restyle(fig, 31, H)
    ax_ = fig.axes[0]
    ax_.set_title(title, fontsize=6)
    ax_.set_ylabel("d2 bin")
    ax_.set_xlabel("Normalized spontaneous\nfiring-rate range")
    h, lab = ax_.get_legend_handles_labels()
    lab2 = [f"{NAME.get(l, l)} {F.stars(DYN[l]['p'])}" for l in lab]
    if ax_.get_legend():
        ax_.get_legend().remove()
    legend_below(ax_, h, lab2, gap_mm=9.2)
    place(fig, 31, (7.0,), 22.0)
    F.settle(fig, tight=False)
    F.drop_counts(fig)
    F.show(F.save(fig, fid, tight=False))